# 03 — Medallion ELT Benchmark on BigQuery DataFrames (`bigframes.pandas`, Serverless BigQuery)

This notebook executes the full **TPC-DS Omnichannel Medallion pipeline** (`SPEC.md` §3) using the official **BigQuery DataFrames (`bigframes.pandas >= 2.40`)** Python API compiled lazily to BigQuery GoogleSQL, executed inside a non-bypassable Enterprise slot reservation (`BQ_SLOT_CAP` slots in `US`).

### Benchmark Design & Architectural Controls
1. **Partial Ordering Mode (`ordering_mode = "partial"`):** Explicitly configured before session creation so `BigFrames` pushes down distributed unordered relational operators directly to BigQuery slots without generating sequential whole-table row-hash ordering indexes.
2. **Dual Storage Backend Passes (Storage vs. Compute Isolation):**
   - **Pass 1 (`bigframes_biglake_gcs` / `biglake_gcs_parquet`):** Reads all 8 Bronze tables (facts + dimensions) from `${PROJECT_ID}.${DS_BRONZE_BIGLAKE}` — BigLake external tables backed by the **exact same GCS Parquet files** read by Dataproc Spark (`01` / `02`).
   - **Pass 2 (`bigframes_native_bq` / `bq_native_capacitor`):** Reads all 8 Bronze tables from `${PROJECT_ID}.${DS_BRONZE_NATIVE}` — native BigQuery Capacitor columnar storage with `SPEC.md` §2 clustering.
3. **Slot Governance Verification:** Verifies live queries execute under `BQ_RESERVATION_NAME` with `total_bytes_billed == 0` in `` `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT `` and tests `SET @@reservation = 'none'; SELECT 1;` truthfully.
4. **Stage 2B Python Execution Contrast (`SQL Pushdown` vs. `@bpd.udf` vs. `@bpd.remote_function`):**
   - **Default (`RUN_BQ_MANAGED_UDF="false"`):** Modular Python scoring rules compile natively into Dremel GoogleSQL pushdown inside the flat-rate slot reservation (`$0` extra SKU, zero JVM/Arrow serialization overhead).
   - **Optional (`RUN_BQ_MANAGED_UDF="true"`):** Executes BigQuery Managed Python UDFs (`@bpd.udf`), billed under the `BigQuery Services` SKU outside slot reservations (unlike `@bpd.remote_function`, which invokes external Cloud Run HTTP endpoints).
5. **Clean Session Lifecycle:** Wraps execution in `try ... finally: bpd.close_session()` so all session-scoped `_anonymous_*` temporary tables/routines are cleaned up immediately.


In [ ]:
import datetime
import os
import re
import sys
import time
import uuid
from pathlib import Path
from typing import Any

import bigframes.pandas as bpd
import google.api_core.exceptions as gapi_exceptions
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

try:
    from IPython.display import display
except ImportError:  # Fallback when running extracted code outside Jupyter/IPython.
    display = print


def resolve_repo_root(start: Path | None = None) -> Path:
    """Walk upward until SPEC.md and pyproject.toml are located."""
    cur = (start or Path.cwd()).resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate repository root above {cur}")


def _parse_bool_env(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def _sanitize_bq_label(value: str, max_len: int = 63) -> str:
    cleaned = re.sub(r"[^a-z0-9_-]", "-", value.strip().lower())
    return (cleaned[:max_len].strip("-_")) or "na"


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env", override=False)

from scripts.metrics_helper import load_benchmark_runs, log_stage_metric

PROJECT_ID = os.getenv("PROJECT_ID", "").strip()
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID must be set to a valid GCP project ID in .env or environment (not 'your-gcp-project-id')."
    )

BQ_LOCATION = os.getenv("BQ_LOCATION", "US").strip()
if not re.fullmatch(r"[A-Za-z0-9_-]+", BQ_LOCATION):
    raise ValueError(f"Invalid BQ_LOCATION: {BQ_LOCATION!r}")
BQ_INFO_SCHEMA_REGION = f"region-{BQ_LOCATION.lower()}"

BQ_ADMIN_PROJECT_ID = os.getenv("BQ_ADMIN_PROJECT_ID", PROJECT_ID).strip() or PROJECT_ID
BQ_RESERVATION_NAME = os.getenv("BQ_RESERVATION_NAME", "bench-elt-slots").strip()
BQ_SLOT_CAP = int(os.getenv("BQ_SLOT_CAP", "100"))
ALLOW_UNRESERVED_BQ = _parse_bool_env("ALLOW_UNRESERVED_BQ", False)
BIGLAKE_CONNECTION_ID = os.getenv("BIGLAKE_CONNECTION_ID", "bench-biglake-us").strip()

DS_BRONZE_BIGLAKE = os.getenv("DS_BRONZE_BIGLAKE", "bench_bronze_biglake").strip()
DS_BRONZE_NATIVE = os.getenv("DS_BRONZE_NATIVE", "bench_bronze_native").strip()
DS_SILVER_BQ = os.getenv("DS_SILVER_BQ", "bench_silver_bq").strip()
DS_GOLD_BQ = os.getenv("DS_GOLD_BQ", "bench_gold_bq").strip()
DS_METRICS = os.getenv("DS_METRICS", "bench_metrics").strip()

SCALE_PRESET = os.getenv("SCALE_PRESET", "small_1g").strip()
if SCALE_PRESET not in {"small_1g", "medium_100g", "large_500g"}:
    raise ValueError(f"Unsupported SCALE_PRESET: {SCALE_PRESET!r}")

RUN_PYTHON_UDF_STAGE = _parse_bool_env("RUN_PYTHON_UDF_STAGE", True)
RUN_BQ_MANAGED_UDF = _parse_bool_env("RUN_BQ_MANAGED_UDF", False)
WRITE_METRICS_TO_BQ = _parse_bool_env("WRITE_METRICS_TO_BQ", True)
METRICS_JSONL_PATH = REPO_ROOT / "results" / "live_runs" / "benchmark_runs.jsonl"

_default_run_id = (
    f"bf-{SCALE_PRESET}-"
    f"{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-"
    f"{uuid.uuid4().hex[:6]}"
)
RUN_ID = (os.getenv("BENCH_RUN_ID") or os.getenv("RUN_ID") or "").strip() or _default_run_id

print(f"Repository root       : {REPO_ROOT}")
print(f"RUN_ID                : {RUN_ID}")
print(f"PROJECT_ID / LOCATION : {PROJECT_ID} / {BQ_LOCATION} ({BQ_INFO_SCHEMA_REGION})")
print(f"Reservation           : {BQ_RESERVATION_NAME} ({BQ_SLOT_CAP} slots | allow_unreserved={ALLOW_UNRESERVED_BQ})")
print(f"SCALE_PRESET          : {SCALE_PRESET}")
print(f"Bronze BigLake / Nat. : {DS_BRONZE_BIGLAKE} / {DS_BRONZE_NATIVE}")
print(f"Silver / Gold / Metr. : {DS_SILVER_BQ} / {DS_GOLD_BQ} / {DS_METRICS}")
print(f"Stage 2B UDF / Managed: {RUN_PYTHON_UDF_STAGE} / {RUN_BQ_MANAGED_UDF}")


In [ ]:
# Configure BigQuery DataFrames BEFORE starting the session so partial ordering applies to all queries.
bpd.close_session()
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = BQ_LOCATION
bpd.options.bigquery.ordering_mode = "partial"

session = bpd.get_global_session()
assert bpd.options.bigquery.ordering_mode == "partial", (
    f"Expected bpd.options.bigquery.ordering_mode == 'partial', got {bpd.options.bigquery.ordering_mode!r}"
)

bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print(
    f"Initialized BigFrames session={session.session_id} | "
    f"ordering_mode={bpd.options.bigquery.ordering_mode!r} | "
    f"bigframes={bpd.__version__}"
)


### Live Slot Reservation Governance Verification (`SPEC.md` §1.4)

Before timing benchmark stages, confirm two properties directly against BigQuery in `US`:
1. Normal analytical queries execute under `BQ_RESERVATION_NAME` with `total_bytes_billed == 0` in `INFORMATION_SCHEMA.JOBS_BY_PROJECT`.
2. Attempting `SET @@reservation = 'none'; SELECT 1;` is rejected (`400`/`403`) for standard job-user principals lacking `bigquery.reservations.*` override permissions (or detects truthfully if the current caller is a Sandbox `roles/owner` admin with override privileges).


In [ ]:
def verify_slot_reservation_governance(
    client: bigquery.Client,
    project_id: str,
    ds_bronze_native: str,
    reservation_name: str,
    info_schema_region: str,
    allow_unreserved_bq: bool = False,
) -> dict[str, Any]:
    """Probe Enterprise reservation assignment and test On-Demand bypass behavior."""
    probe_sql = (
        f"SELECT COUNT(*) AS row_cnt, MIN(d_date_sk) AS min_sk, MAX(d_date_sk) AS max_sk "
        f"FROM `{project_id}.{ds_bronze_native}.date_dim`"
    )
    probe_job = client.query(probe_sql, job_config=bigquery.QueryJobConfig(use_query_cache=False))
    list(probe_job.result())

    audit_sql = f"""
    SELECT
      job_id,
      COALESCE(reservation_id, 'none') AS reservation_id,
      COALESCE(total_bytes_billed, 0) AS total_bytes_billed,
      COALESCE(total_bytes_processed, 0) AS total_bytes_processed,
      COALESCE(total_slot_ms, 0) AS total_slot_ms
    FROM `{info_schema_region}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 10 MINUTE)
      AND job_id = @probe_job_id
    """
    audit_cfg = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("probe_job_id", "STRING", probe_job.job_id)],
        use_query_cache=False,
    )
    audit_rows = list(client.query(audit_sql, job_config=audit_cfg).result())
    if audit_rows:
        r = audit_rows[0]
        res_id = str(r.reservation_id)
        bytes_billed = int(r.total_bytes_billed)
        slot_ms = int(r.total_slot_ms)
    else:
        raw_res = getattr(probe_job, "reservation_id", None) or (
            getattr(probe_job, "_properties", {}) or {}
        ).get("statistics", {}).get("reservation_id")
        res_id = str(raw_res or "none")
        bytes_billed = int(probe_job.total_bytes_billed or 0)
        slot_ms = int(probe_job.slot_millis or 0)

    in_expected_res = (reservation_name in res_id) and (res_id != "none")
    zero_on_demand_billed = (bytes_billed == 0)

    print(
        f"[Governance Probe] job_id={probe_job.job_id} | reservation_id={res_id} | "
        f"total_bytes_billed={bytes_billed} | total_slot_ms={slot_ms}"
    )
    if not in_expected_res or not zero_on_demand_billed:
        msg = (
            f"Probe query did not execute inside expected reservation {reservation_name!r} "
            f"with total_bytes_billed == 0 (observed reservation_id={res_id!r}, "
            f"total_bytes_billed={bytes_billed})."
        )
        if not allow_unreserved_bq:
            raise RuntimeError(
                f"{msg} Failing closed to prevent unfair unreserved On-Demand execution; "
                "fix reservation assignment or explicitly set ALLOW_UNRESERVED_BQ='true'."
            )
        print(f"[WARN] ALLOW_UNRESERVED_BQ=true: {msg}")
    else:
        print("[PASS] Probe query executed inside assigned slot reservation with $0 On-Demand billing.")

    # Test On-Demand bypass behavior truthfully without faking errors.
    bypass_status: str
    bypass_detail: str
    try:
        bypass_job = client.query(
            "SET @@reservation = 'none'; SELECT 1 AS bypass_probe;",
            job_config=bigquery.QueryJobConfig(use_query_cache=False),
        )
        list(bypass_job.result())
        bypass_res = getattr(bypass_job, "reservation_id", None) or "none"
        bypass_status = "WARN_SANDBOX_OVERRIDE_PRIVILEGE"
        bypass_detail = (
            f"Caller succeeded with SET @@reservation='none' (job_id={bypass_job.job_id}, "
            f"reservation_id={bypass_res}). This occurs when executing as a Sandbox roles/owner "
            "admin holding bigquery.reservations.* override permissions; production workloads must "
            "use roles/bigquery.jobUser (without reservation admin) to make slot caps non-bypassable."
        )
        print(f"[WARN] {bypass_detail}")
    except (gapi_exceptions.BadRequest, gapi_exceptions.Forbidden) as exc:
        bypass_status = "PASS_BYPASS_REJECTED"
        err_msg = getattr(exc, "message", None) or str(exc)
        bypass_detail = f"On-Demand reservation bypass rejected ({type(exc).__name__}): {err_msg}"
        print(f"[PASS] {bypass_detail}")

    return {
        "probe_job_id": probe_job.job_id,
        "reservation_id": res_id,
        "total_bytes_billed": bytes_billed,
        "in_expected_reservation": in_expected_res,
        "zero_bytes_billed": zero_on_demand_billed,
        "bypass_test_status": bypass_status,
        "bypass_test_detail": bypass_detail,
    }


In [ ]:
def read_gbq_table_lazy(
    bf_session: Any,
    table_fqn: str,
    columns: list[str] | None = None,
) -> bpd.DataFrame:
    """Read a BigQuery Native or BigLake External table lazily without metadata cache reuse.

    BigQuery does not support FOR SYSTEM_TIME AS OF snapshots on standard GCS external
    tables; bigframes 2.53.0's public ``read_gbq_table`` time-travel check only catches
    ``NotFound``, so if dry-run validation raises ``400 BadRequest`` for BigLake tables,
    retry lazily via ``_loader.read_gbq_table(..., enable_snapshot=False)``.
    """
    cols = columns or ()
    try:
        return bpd.read_gbq_table(table_fqn, columns=cols, use_cache=False)
    except gapi_exceptions.BadRequest as exc:
        msg = str(exc).lower()
        if any(tok in msg for tok in ("time travel", "system_time", "external", "snapshot")):
            return bf_session._loader.read_gbq_table(
                table_id=table_fqn,
                columns=cols,
                use_cache=False,
                enable_snapshot=False,
            )
        raise


def get_session_job_ids(bf_session: Any) -> list[str]:
    """Return ordered BigQuery job_ids currently recorded in BigFrames session history."""
    hist_df = bf_session.execution_history(all_cells=True).to_dataframe()
    if hist_df.empty or "job_id" not in hist_df.columns:
        return []
    return [str(jid) for jid in hist_df["job_id"].dropna().tolist() if str(jid).strip()]


def query_stage_bq_telemetry(
    client: bigquery.Client,
    info_schema_region: str,
    start_epoch_sec: float,
    end_epoch_sec: float,
    stage_job_ids: list[str],
    stage_label_key: str,
    stage_label_val: str,
) -> dict[str, Any]:
    """Extract slot_ms, bytes_processed, reservation_id, and BI Engine telemetry for a stage."""
    params: list[Any] = [
        bigquery.ScalarQueryParameter("start_epoch", "INT64", int(start_epoch_sec) - 10),
        bigquery.ScalarQueryParameter("end_epoch", "INT64", int(end_epoch_sec) + 30),
    ]
    if stage_job_ids:
        filter_clause = "AND job_id IN UNNEST(@stage_job_ids)"
        params.append(bigquery.ArrayQueryParameter("stage_job_ids", "STRING", stage_job_ids))
    else:
        filter_clause = (
            "AND job_type = 'QUERY' "
            "AND EXISTS (SELECT 1 FROM UNNEST(labels) AS lbl "
            "WHERE lbl.key = @lbl_key AND lbl.value = @lbl_val)"
        )
        params.extend(
            [
                bigquery.ScalarQueryParameter("lbl_key", "STRING", stage_label_key),
                bigquery.ScalarQueryParameter("lbl_val", "STRING", stage_label_val),
            ]
        )

    sql = f"""
    WITH stage_jobs AS (
      SELECT
        job_id,
        COALESCE(total_slot_ms, 0) AS total_slot_ms,
        COALESCE(total_bytes_processed, 0) AS total_bytes_processed,
        COALESCE(total_bytes_billed, 0) AS total_bytes_billed,
        NULLIF(TRIM(reservation_id), '') AS reservation_id,
        COALESCE(bi_engine_statistics.bi_engine_mode, 'NONE') AS bi_engine_mode,
        bi_engine_statistics.bi_engine_reasons AS bi_engine_reasons
      FROM `{info_schema_region}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
      WHERE creation_time BETWEEN TIMESTAMP_SECONDS(@start_epoch) AND TIMESTAMP_SECONDS(@end_epoch)
        AND user_email = SESSION_USER()
        {filter_clause}
    ),
    agg_stats AS (
      SELECT
        COUNT(*) AS job_count,
        COALESCE(SUM(total_slot_ms), 0) AS bq_total_slot_ms,
        COALESCE(SUM(total_bytes_processed), 0) AS bq_bytes_processed,
        COALESCE(SUM(total_bytes_billed), 0) AS bq_bytes_billed,
        ARRAY_AGG(DISTINCT reservation_id IGNORE NULLS) AS reservation_ids,
        ARRAY_AGG(DISTINCT bi_engine_mode IGNORE NULLS) AS bi_engine_modes
      FROM stage_jobs
    ),
    reason_codes AS (
      SELECT ARRAY_AGG(DISTINCT r.code IGNORE NULLS) AS bi_engine_reason_codes
      FROM stage_jobs AS sj, UNNEST(sj.bi_engine_reasons) AS r
    )
    SELECT
      agg_stats.job_count,
      agg_stats.bq_total_slot_ms,
      agg_stats.bq_bytes_processed,
      agg_stats.bq_bytes_billed,
      agg_stats.reservation_ids,
      agg_stats.bi_engine_modes,
      reason_codes.bi_engine_reason_codes
    FROM agg_stats
    CROSS JOIN reason_codes
    """

    # Retry briefly outside the stage timer for INFORMATION_SCHEMA eventual visibility.
    row = None
    expected_min = len(stage_job_ids) if stage_job_ids else 1
    for attempt in range(3):
        rows = list(
            client.query(
                sql,
                job_config=bigquery.QueryJobConfig(query_parameters=params, use_query_cache=False),
            ).result()
        )
        row = rows[0] if rows else None
        if row is not None and int(row.job_count) >= expected_min:
            break
        if attempt < 2:
            time.sleep(2.0)

    if row is not None and int(row.job_count) > 0:
        res_list = list(row.reservation_ids or [])
        bi_modes = list(row.bi_engine_modes or [])
        bi_reasons = list(row.bi_engine_reason_codes or [])
        return {
            "job_count": int(row.job_count),
            "bq_total_slot_ms": int(row.bq_total_slot_ms),
            "bq_bytes_processed": int(row.bq_bytes_processed),
            "bq_bytes_billed": int(row.bq_bytes_billed),
            "reservation_id": ", ".join(res_list) if res_list else "none",
            "bi_engine_mode": ", ".join(bi_modes) if bi_modes else "NONE",
            "bi_engine_reasons": ", ".join(bi_reasons) if bi_reasons else "",
        }

    # Defensive fallback via Jobs API if INFORMATION_SCHEMA lags behind job completion.
    total_slot_ms = 0
    total_bytes_processed = 0
    total_bytes_billed = 0
    res_set: set[str] = set()
    for jid in stage_job_ids:
        job = client.get_job(jid)
        total_slot_ms += int(getattr(job, "slot_millis", 0) or 0)
        total_bytes_processed += int(getattr(job, "total_bytes_processed", 0) or 0)
        total_bytes_billed += int(getattr(job, "total_bytes_billed", 0) or 0)
        rid = getattr(job, "reservation_id", None) or (
            getattr(job, "_properties", {}) or {}
        ).get("statistics", {}).get("reservation_id")
        if rid:
            res_set.add(str(rid))
    return {
        "job_count": len(stage_job_ids),
        "bq_total_slot_ms": total_slot_ms,
        "bq_bytes_processed": total_bytes_processed,
        "bq_bytes_billed": total_bytes_billed,
        "reservation_id": ", ".join(sorted(res_set)) if res_set else "unknown",
        "bi_engine_mode": "UNKNOWN",
        "bi_engine_reasons": "",
    }


def fetch_materialized_checksum(
    client: bigquery.Client,
    table_fqn: str,
    checksum_col: str,
) -> tuple[int, float]:
    """Compute exact output row count and ROUND(SUM(checksum_col), 2) outside timers."""
    sql = f"""
    SELECT
      COUNT(*) AS output_row_count,
      ROUND(COALESCE(SUM(CAST({checksum_col} AS FLOAT64)), 0.0), 2) AS checksum_net_paid
    FROM `{table_fqn}`
    """
    rows = list(client.query(sql, job_config=bigquery.QueryJobConfig(use_query_cache=False)).result())
    row = rows[0]
    return int(row.output_row_count), float(row.checksum_net_paid or 0.0)


### Medallion Pipeline Stage Definitions (`SPEC.md` §3 Parity Contract)

All 4 stages below are parameterized by storage backend (`bronze_ds`, `silver_ds`, `gold_ds`, `suffix`) and implemented using genuine `bigframes.pandas` DataFrame/Series operations compiled lazily to BigQuery SQL in `ordering_mode="partial"`:
- **Stage 1 (`stage1_bronze_to_silver`):** Pre-aggregates `store_returns` and `web_returns` BEFORE left-joining `store_sales` and `web_sales` (anti-fan-out rule), filters clean sales lines (`sold_date_sk`, `item_sk`, `customer_sk` non-null, `quantity > 0`), unions via `bpd.concat(..., ignore_index=True)`, and materializes 15 Silver columns.
- **Stage 2A (`stage2a_silver_vectorized`):** Computes `order_risk_tier` (`case_when` 4-branch predicate) and `adjusted_margin_score` rounded to 4 decimals via vectorized expressions.
- **Stage 2B (`stage2b_silver_udf`):** Deterministic sample filter (`100%` on `small_1g`, `(order_number.abs() % 20) == 0` 5% on `medium_100g`/`large_500g`) contrasting native Dremel SQL pushdown inside the slot reservation vs. optional `@bpd.udf` Managed Python UDF containers.
- **Stage 3 (`stage3_silver_to_gold`):** Joins materialized `silver_enriched_sales_*` with the 4 dimension tables **from the matching storage backend** (`DS_BRONZE_BIGLAKE` or `DS_BRONZE_NATIVE`), filters NULL join keys prior to `.merge(...)` for exact SQL `INNER JOIN` parity, caches the daily aggregate (`daily.cache()`) prior to 7d/30d `rolling` windows and `category_daily_share`, and materializes `gold_omnichannel_kpi_mart_*`.


In [ ]:
STAGE1_SILVER_COLUMNS = [
    "channel",
    "sold_date_sk",
    "item_sk",
    "customer_sk",
    "cdemo_sk",
    "order_number",
    "quantity",
    "gross_sales_price",
    "discount_amt",
    "net_paid",
    "net_profit",
    "return_qty",
    "return_amt",
    "net_paid_after_return",
    "net_profit_after_return",
]


def _build_channel_silver(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    *,
    channel_label: str,
    sales_table: str,
    returns_table: str,
    s_prefix: str,
    r_prefix: str,
    customer_col: str,
    cdemo_col: str,
    order_col: str,
    ret_order_col: str,
) -> bpd.DataFrame:
    """Normalize one TPC-DS sales+returns channel into the 15-column Stage 1 schema."""
    sold_col = f"{s_prefix}sold_date_sk"
    item_col = f"{s_prefix}item_sk"
    qty_col = f"{s_prefix}quantity"
    ext_price_col = f"{s_prefix}ext_sales_price"
    ext_disc_col = f"{s_prefix}ext_discount_amt"
    net_paid_col = f"{s_prefix}net_paid"
    net_profit_col = f"{s_prefix}net_profit"

    r_item_col = f"{r_prefix}item_sk"
    r_qty_col = f"{r_prefix}return_quantity"
    r_amt_col = f"{r_prefix}return_amt"
    r_loss_col = f"{r_prefix}net_loss"

    sales_df = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.{sales_table}",
        columns=[
            sold_col,
            item_col,
            customer_col,
            cdemo_col,
            order_col,
            qty_col,
            ext_price_col,
            ext_disc_col,
            net_paid_col,
            net_profit_col,
        ],
    )
    sales_clean = sales_df[
        sales_df[sold_col].notna()
        & sales_df[item_col].notna()
        & sales_df[customer_col].notna()
        & (sales_df[qty_col] > 0)
    ]

    returns_df = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.{returns_table}",
        columns=[r_item_col, ret_order_col, r_qty_col, r_amt_col, r_loss_col],
    )
    returns_clean = returns_df[
        returns_df[r_item_col].notna() & returns_df[ret_order_col].notna()
    ].assign(
        **{
            r_qty_col: returns_df[r_qty_col].fillna(0),
            r_amt_col: returns_df[r_amt_col].fillna(0.0),
            r_loss_col: returns_df[r_loss_col].fillna(0.0),
        }
    )
    returns_agg = returns_clean.groupby(
        [r_item_col, ret_order_col], as_index=False, dropna=False
    ).agg(
        _agg_return_qty=(r_qty_col, "sum"),
        _agg_return_amt=(r_amt_col, "sum"),
        _agg_return_net_loss=(r_loss_col, "sum"),
    )

    joined = sales_clean.merge(
        returns_agg,
        how="left",
        left_on=[item_col, order_col],
        right_on=[r_item_col, ret_order_col],
    )

    normalized = joined.assign(
        channel=channel_label,
        sold_date_sk=joined[sold_col].astype("Int64"),
        item_sk=joined[item_col].astype("Int64"),
        customer_sk=joined[customer_col].astype("Int64"),
        cdemo_sk=joined[cdemo_col].astype("Int64"),
        order_number=joined[order_col].astype("Int64"),
        quantity=joined[qty_col].astype("Int64"),
        gross_sales_price=joined[ext_price_col].fillna(0.0).astype("Float64"),
        discount_amt=joined[ext_disc_col].fillna(0.0).astype("Float64"),
        net_paid=joined[net_paid_col].fillna(0.0).astype("Float64"),
        net_profit=joined[net_profit_col].fillna(0.0).astype("Float64"),
        return_qty=joined["_agg_return_qty"].fillna(0).astype("Int64"),
        return_amt=joined["_agg_return_amt"].fillna(0.0).astype("Float64"),
        _return_net_loss=joined["_agg_return_net_loss"].fillna(0.0).astype("Float64"),
    )
    normalized = normalized.assign(
        net_paid_after_return=(normalized["net_paid"] - normalized["return_amt"]).astype("Float64"),
        net_profit_after_return=(
            normalized["net_profit"] - normalized["_return_net_loss"]
        ).astype("Float64"),
    )
    return normalized[STAGE1_SILVER_COLUMNS]


def stage1_bronze_to_silver(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    silver_ds: str,
    suffix: str,
    labels: dict[str, str],
) -> tuple[str, float]:
    """Stage 1: Deduplicated store + web sales with pre-aggregated returns -> Silver."""
    store_df = _build_channel_silver(
        bf_session,
        project_id,
        bronze_ds,
        channel_label="STORE",
        sales_table="store_sales",
        returns_table="store_returns",
        s_prefix="ss_",
        r_prefix="sr_",
        customer_col="ss_customer_sk",
        cdemo_col="ss_cdemo_sk",
        order_col="ss_ticket_number",
        ret_order_col="sr_ticket_number",
    )
    web_df = _build_channel_silver(
        bf_session,
        project_id,
        bronze_ds,
        channel_label="WEB",
        sales_table="web_sales",
        returns_table="web_returns",
        s_prefix="ws_",
        r_prefix="wr_",
        customer_col="ws_bill_customer_sk",
        cdemo_col="ws_bill_cdemo_sk",
        order_col="ws_order_number",
        ret_order_col="wr_order_number",
    )
    silver_df = bpd.concat([store_df, web_df], ignore_index=True)
    dest_table = f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}"
    silver_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0


In [ ]:
def apply_vectorized_scoring(df: bpd.DataFrame) -> bpd.DataFrame:
    """Apply SPEC.md Stage 2A/2B scoring rules cleanly as compiled Dremel GoogleSQL pushdown."""
    high_risk = (df["return_amt"] > 0.5 * df["gross_sales_price"]) & (df["return_qty"] > 0)
    vip_full = (df["net_profit_after_return"] > 50.0) & (df["discount_amt"] == 0.0)
    promo = df["discount_amt"] > 0.25 * df["gross_sales_price"]

    order_risk_tier = df["channel"].case_when(
        [
            (high_risk, "HIGH_RETURN_RISK"),
            (vip_full, "VIP_FULL_PRICE"),
            (promo, "PROMO_DRIVEN"),
            (True, "STANDARD"),
        ]
    )
    adjusted_margin_score = (
        df["net_profit_after_return"]
        - (0.15 * df["return_amt"])
        + (0.05 * df["net_paid_after_return"])
    ).round(4)
    return df.assign(
        order_risk_tier=order_risk_tier,
        adjusted_margin_score=adjusted_margin_score,
    )


def stage2a_silver_vectorized(
    bf_session: Any,
    project_id: str,
    silver_ds: str,
    suffix: str,
    labels: dict[str, str],
) -> tuple[str, float]:
    """Stage 2A: Pure vectorized SQL pushdown enrichment on materialized Silver."""
    silver_src = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}",
    )
    enriched = apply_vectorized_scoring(silver_src)
    dest_table = f"{project_id}.{silver_ds}.silver_enriched_sales_{suffix}"
    enriched.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0


In [ ]:
def _apply_bq_managed_python_udf_scoring(
    df: bpd.DataFrame,
    silver_ds: str,
) -> bpd.DataFrame:
    """Optional Stage 2B toggle: execute BigQuery Managed Python UDFs via @bpd.udf.

    Unlike native Dremel SQL pushdown (which executes strictly inside BQ_RESERVATION_NAME
    at $0 extra SKU) and @bpd.remote_function (which calls an external Cloud Run HTTP
    service), @bpd.udf deploys container-backed BigQuery Managed Python routines billed
    separately under the BigQuery Services SKU outside slot reservations.
    """

    @bpd.udf(
        input_types=[float, float, int, float, float],
        output_type=str,
        dataset=silver_ds,
    )
    def udf_order_risk_tier(
        return_amt: float,
        gross_sales_price: float,
        return_qty: int,
        net_profit_after_return: float,
        discount_amt: float,
    ) -> str:
        r_amt = float(return_amt or 0.0)
        gross = float(gross_sales_price or 0.0)
        r_qty = int(return_qty or 0)
        net_prof = float(net_profit_after_return or 0.0)
        disc = float(discount_amt or 0.0)
        if r_amt > 0.5 * gross and r_qty > 0:
            return "HIGH_RETURN_RISK"
        elif net_prof > 50.0 and disc == 0.0:
            return "VIP_FULL_PRICE"
        elif disc > 0.25 * gross:
            return "PROMO_DRIVEN"
        return "STANDARD"

    @bpd.udf(
        input_types=[float, float, float],
        output_type=float,
        dataset=silver_ds,
    )
    def udf_adjusted_margin_score(
        net_profit_after_return: float,
        return_amt: float,
        net_paid_after_return: float,
    ) -> float:
        net_prof = float(net_profit_after_return or 0.0)
        r_amt = float(return_amt or 0.0)
        net_paid = float(net_paid_after_return or 0.0)
        return round(net_prof - (0.15 * r_amt) + (0.05 * net_paid), 4)

    tier_inputs = df[
        ["return_amt", "gross_sales_price", "return_qty", "net_profit_after_return", "discount_amt"]
    ].assign(
        return_amt=df["return_amt"].astype("Float64"),
        gross_sales_price=df["gross_sales_price"].astype("Float64"),
        return_qty=df["return_qty"].astype("Int64"),
        net_profit_after_return=df["net_profit_after_return"].astype("Float64"),
        discount_amt=df["discount_amt"].astype("Float64"),
    )
    margin_inputs = df[
        ["net_profit_after_return", "return_amt", "net_paid_after_return"]
    ].assign(
        net_profit_after_return=df["net_profit_after_return"].astype("Float64"),
        return_amt=df["return_amt"].astype("Float64"),
        net_paid_after_return=df["net_paid_after_return"].astype("Float64"),
    )
    tier_series = tier_inputs.apply(udf_order_risk_tier, axis=1)
    margin_series = margin_inputs.apply(udf_adjusted_margin_score, axis=1)
    return df.assign(order_risk_tier=tier_series, adjusted_margin_score=margin_series)


def stage2b_silver_udf(
    bf_session: Any,
    project_id: str,
    silver_ds: str,
    suffix: str,
    scale_preset: str,
    run_bq_managed_udf: bool,
    labels: dict[str, str],
) -> tuple[str, float]:
    """Stage 2B: Deterministic row sampling + Python business rules (SQL pushdown vs @bpd.udf)."""
    silver_src = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_omnichannel_sales_{suffix}",
    )
    if scale_preset == "small_1g":
        sampled = silver_src
        sample_fraction = 1.0
    else:
        sampled = silver_src[(silver_src["order_number"].abs() % 20) == 0]
        sample_fraction = 0.05

    if run_bq_managed_udf:
        out_df = _apply_bq_managed_python_udf_scoring(sampled, silver_ds=silver_ds)
    else:
        out_df = apply_vectorized_scoring(sampled)

    dest_table = f"{project_id}.{silver_ds}.silver_udf_benchmark_out_{suffix}"
    out_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, sample_fraction


In [ ]:
STAGE3_GOLD_COLUMNS = [
    "d_date_sk",
    "d_date",
    "d_year",
    "d_moy",
    "channel",
    "i_category",
    "cd_credit_rating",
    "order_risk_tier",
    "order_lines",
    "unique_customers",
    "daily_gross_sales",
    "daily_return_amt",
    "daily_net_paid",
    "daily_net_profit",
    "avg_margin_score",
    "rolling_7d_net_paid",
    "rolling_30d_net_paid",
    "rolling_30d_return_rate",
    "category_daily_share",
]


def stage3_silver_to_gold(
    bf_session: Any,
    project_id: str,
    bronze_ds: str,
    silver_ds: str,
    gold_ds: str,
    suffix: str,
    labels: dict[str, str],
) -> tuple[str, float]:
    """Stage 3: 5-way Star Schema join against matching Bronze storage + rolling KPI windows."""
    enriched = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{silver_ds}.silver_enriched_sales_{suffix}",
        columns=[
            "channel",
            "sold_date_sk",
            "item_sk",
            "customer_sk",
            "cdemo_sk",
            "gross_sales_price",
            "return_amt",
            "net_paid_after_return",
            "net_profit_after_return",
            "order_risk_tier",
            "adjusted_margin_score",
        ],
    )
    customer = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.customer",
        columns=["c_customer_sk", "c_current_cdemo_sk"],
    )
    cust_demo = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.customer_demographics",
        columns=["cd_demo_sk", "cd_credit_rating"],
    )
    item = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.item",
        columns=["i_item_sk", "i_category"],
    )
    date_dim = read_gbq_table_lazy(
        bf_session,
        f"{project_id}.{bronze_ds}.date_dim",
        columns=["d_date_sk", "d_date", "d_year", "d_moy"],
    )

    # Filter non-null join keys and filter predicates before merging so BigFrames achieves
    # 100% SQL INNER JOIN semantics (bpd.DataFrame.merge otherwise treats NULL == NULL as a match).
    enriched = enriched[
        enriched["customer_sk"].notna()
        & enriched["item_sk"].notna()
        & enriched["sold_date_sk"].notna()
    ]
    customer = customer[customer["c_customer_sk"].notna()]
    cust_demo = cust_demo[
        cust_demo["cd_demo_sk"].notna() & cust_demo["cd_credit_rating"].notna()
    ]
    item = item[item["i_item_sk"].notna() & item["i_category"].notna()]
    date_dim = date_dim[date_dim["d_date_sk"].notna() & date_dim["d_date"].notna()]

    joined = enriched.merge(
        customer, how="inner", left_on="customer_sk", right_on="c_customer_sk"
    )
    joined = joined.assign(
        effective_cdemo_sk=joined["cdemo_sk"].fillna(joined["c_current_cdemo_sk"])
    )
    joined = joined[joined["effective_cdemo_sk"].notna()]
    joined = joined.merge(
        cust_demo, how="inner", left_on="effective_cdemo_sk", right_on="cd_demo_sk"
    )
    joined = joined.merge(item, how="inner", left_on="item_sk", right_on="i_item_sk")
    joined = joined.merge(date_dim, how="inner", left_on="sold_date_sk", right_on="d_date_sk")

    grain_cols = [
        "d_date_sk",
        "d_date",
        "d_year",
        "d_moy",
        "channel",
        "i_category",
        "cd_credit_rating",
        "order_risk_tier",
    ]
    daily = joined.groupby(grain_cols, as_index=False, dropna=False).agg(
        order_lines=("channel", "count"),
        unique_customers=("customer_sk", "nunique"),
        daily_gross_sales=("gross_sales_price", "sum"),
        daily_return_amt=("return_amt", "sum"),
        daily_net_paid=("net_paid_after_return", "sum"),
        daily_net_profit=("net_profit_after_return", "sum"),
        avg_margin_score=("adjusted_margin_score", "mean"),
    )

    # Materialize compact daily aggregate inside the Stage 3 timer so rolling windows and
    # daily channel totals scan the aggregated temp table instead of repeating the 5-way join.
    daily = daily.cache()

    part_cols = ["channel", "i_category", "cd_credit_rating", "order_risk_tier"]
    d_idx = daily.sort_values("d_date_sk", ascending=True).set_index(
        ["d_date_sk", *part_cols], drop=True
    )
    r7 = (
        d_idx.groupby(part_cols, dropna=False)["daily_net_paid"]
        .rolling(7, min_periods=1)
        .sum()
        .droplevel([0, 1, 2, 3])
    )
    r30 = (
        d_idx.groupby(part_cols, dropna=False)[
            ["daily_net_paid", "daily_return_amt", "daily_gross_sales"]
        ]
        .rolling(30, min_periods=1)
        .sum()
        .droplevel([0, 1, 2, 3])
    )
    d_win = d_idx.assign(
        rolling_7d_net_paid=r7,
        rolling_30d_net_paid=r30["daily_net_paid"],
        _r30_return_amt=r30["daily_return_amt"],
        _r30_gross_sales=r30["daily_gross_sales"],
    ).reset_index(drop=False)

    chan_daily = daily.groupby(["d_date_sk", "channel"], as_index=False, dropna=False).agg(
        _channel_daily_net_paid=("daily_net_paid", "sum"),
    )
    gold_df = d_win.merge(chan_daily, how="inner", on=["d_date_sk", "channel"])
    gold_df = gold_df.assign(
        rolling_30d_return_rate=(
            gold_df["_r30_return_amt"]
            / gold_df["_r30_gross_sales"].where(gold_df["_r30_gross_sales"] != 0)
        ).astype("Float64"),
        category_daily_share=(
            gold_df["daily_net_paid"]
            / gold_df["_channel_daily_net_paid"].where(gold_df["_channel_daily_net_paid"] != 0)
        ).astype("Float64"),
    )[STAGE3_GOLD_COLUMNS]

    dest_table = f"{project_id}.{gold_ds}.gold_omnichannel_kpi_mart_{suffix}"
    gold_df.to_gbq(dest_table, if_exists="replace", index=False, labels=labels)
    return dest_table, 1.0


In [ ]:
def run_medallion_pass(
    *,
    bf_session: Any,
    client: bigquery.Client,
    engine_variant: str,
    storage_backend: str,
    bronze_ds: str,
    silver_ds: str,
    gold_ds: str,
    suffix: str,
) -> list[dict[str, Any]]:
    """Execute Stages 1 -> 2A -> (optional 2B) -> 3 for one Bronze storage backend."""
    print(
        f"\n=== Starting {engine_variant} ({storage_backend}) | "
        f"bronze={bronze_ds} -> silver={silver_ds}, gold={gold_ds} (suffix=_{suffix}) ==="
    )

    stage_specs = [
        (
            "stage1_bronze_to_silver",
            "net_paid_after_return",
            lambda lbl: stage1_bronze_to_silver(
                bf_session, PROJECT_ID, bronze_ds, silver_ds, suffix, lbl
            ),
            "",
        ),
        (
            "stage2a_silver_vectorized",
            "net_paid_after_return",
            lambda lbl: stage2a_silver_vectorized(
                bf_session, PROJECT_ID, silver_ds, suffix, lbl
            ),
            "",
        ),
    ]
    if RUN_PYTHON_UDF_STAGE:
        stage2b_note = (
            "bq_managed_python_udf_outside_reservation"
            if RUN_BQ_MANAGED_UDF
            else "sql_pushdown_in_reservation"
        )
        stage_specs.append(
            (
                "stage2b_silver_udf",
                "net_paid_after_return",
                lambda lbl: stage2b_silver_udf(
                    bf_session,
                    PROJECT_ID,
                    silver_ds,
                    suffix,
                    SCALE_PRESET,
                    RUN_BQ_MANAGED_UDF,
                    lbl,
                ),
                stage2b_note,
            )
        )
    stage_specs.append(
        (
            "stage3_silver_to_gold",
            "daily_net_paid",
            lambda lbl: stage3_silver_to_gold(
                bf_session, PROJECT_ID, bronze_ds, silver_ds, gold_ds, suffix, lbl
            ),
            "",
        )
    )

    results: list[dict[str, Any]] = []
    silver_row_count: int | None = None
    for stage_id, checksum_col, stage_fn, stage_note in stage_specs:
        stage_tag = _sanitize_bq_label(f"{RUN_ID}-{engine_variant}-{stage_id}")
        stage_labels = {
            "bench_run_id": _sanitize_bq_label(RUN_ID),
            "engine_variant": _sanitize_bq_label(engine_variant),
            "stage_id": _sanitize_bq_label(stage_id),
            "bench_stage_key": stage_tag,
        }
        jobs_before = set(get_session_job_ids(bf_session))
        bpd.options.compute.extra_query_labels.clear()
        bpd.options.compute.assign_extra_query_labels(**stage_labels)

        start_epoch_sec = time.time()
        try:
            dest_table, nominal_sample_fraction = stage_fn(stage_labels)
        finally:
            bpd.options.compute.extra_query_labels.clear()
        end_epoch_sec = time.time()

        jobs_after = get_session_job_ids(bf_session)
        stage_job_ids = [jid for jid in jobs_after if jid not in jobs_before]

        # Post-stage telemetry and verification execute STRICTLY outside the timed block.
        bq_meta = query_stage_bq_telemetry(
            client=client,
            info_schema_region=BQ_INFO_SCHEMA_REGION,
            start_epoch_sec=start_epoch_sec,
            end_epoch_sec=end_epoch_sec,
            stage_job_ids=stage_job_ids,
            stage_label_key="bench_stage_key",
            stage_label_val=stage_tag,
        )
        assert bq_meta["job_count"] > 0, (
            f"Telemetry lookup returned 0 BigQuery jobs for {engine_variant} / {stage_id}"
        )

        in_stage_res = (
            (BQ_RESERVATION_NAME in bq_meta["reservation_id"])
            and (bq_meta["reservation_id"] not in {"none", "unknown"})
        )
        allow_managed_udf_billing = (stage_id == "stage2b_silver_udf") and RUN_BQ_MANAGED_UDF
        if not in_stage_res:
            if not ALLOW_UNRESERVED_BQ:
                raise RuntimeError(
                    f"Stage {engine_variant} / {stage_id} executed outside expected reservation "
                    f"{BQ_RESERVATION_NAME!r}: observed reservation_id={bq_meta['reservation_id']!r}."
                )
            effective_bq_slot_cap = None
            unreserved_tag = "unreserved_on_demand"
            base_note = f"{stage_note}; {unreserved_tag}" if stage_note else unreserved_tag
        else:
            if not ALLOW_UNRESERVED_BQ and not allow_managed_udf_billing:
                assert bq_meta["bq_bytes_billed"] == 0, (
                    f"Expected total_bytes_billed == 0 under slot reservation for {stage_id}, "
                    f"got {bq_meta['bq_bytes_billed']}"
                )
            effective_bq_slot_cap = BQ_SLOT_CAP
            base_note = stage_note

        output_row_count, checksum_net_paid = fetch_materialized_checksum(
            client=client,
            table_fqn=dest_table,
            checksum_col=checksum_col,
        )
        assert output_row_count > 0, f"{stage_id} materialized 0 rows into {dest_table}"

        if stage_id == "stage1_bronze_to_silver":
            silver_row_count = output_row_count
        if stage_id == "stage2b_silver_udf":
            if silver_row_count is None:
                silver_src_fqn = f"{PROJECT_ID}.{silver_ds}.silver_omnichannel_sales_{suffix}"
                silver_row_count, _ = fetch_materialized_checksum(
                    client=client,
                    table_fqn=silver_src_fqn,
                    checksum_col="net_paid_after_return",
                )
            assert silver_row_count > 0, "Expected non-zero Silver row count before Stage 2B"
            sample_fraction = round(output_row_count / float(silver_row_count), 6)
        else:
            sample_fraction = nominal_sample_fraction

        note = base_note
        if bq_meta["bi_engine_mode"] not in {"NONE", "UNKNOWN"}:
            bi_tag = f"bi_engine={bq_meta['bi_engine_mode']}"
            note = f"{note}; {bi_tag}" if note else bi_tag

        record = log_stage_metric(
            run_id=RUN_ID,
            scale_preset=SCALE_PRESET,
            engine_variant=engine_variant,
            storage_backend=storage_backend,
            stage_id=stage_id,
            start_epoch_sec=start_epoch_sec,
            end_epoch_sec=end_epoch_sec,
            output_row_count=output_row_count,
            checksum_net_paid=checksum_net_paid,
            worker_machine_type=None,
            total_worker_vcpus=None,
            bq_slot_cap=effective_bq_slot_cap,
            bq_total_slot_ms=bq_meta["bq_total_slot_ms"],
            bq_bytes_processed=bq_meta["bq_bytes_processed"],
            nqe_enabled=False,
            nqe_fallback_note=note,
            est_list_cost_usd=None,
            sample_fraction=sample_fraction,
            jsonl_path=METRICS_JSONL_PATH,
            write_to_bq=WRITE_METRICS_TO_BQ,
            project_id=PROJECT_ID,
            ds_metrics=DS_METRICS,
        )
        combined = {**record, **bq_meta, "dest_table": dest_table}
        results.append(combined)
        print(
            f"[{engine_variant} | {stage_id}] "
            f"wall_clock={record['wall_clock_sec']:.2f}s | rows={output_row_count:,} | "
            f"sample_frac={record['sample_fraction']:.6f} | "
            f"checksum={checksum_net_paid:,.2f} | slot_ms={bq_meta['bq_total_slot_ms']:,} | "
            f"bytes={bq_meta['bq_bytes_processed']:,} | res={bq_meta['reservation_id']} | "
            f"bi={bq_meta['bi_engine_mode']}"
        )
    return results


### Execute Governance Verification + Pass 1 (BigLake GCS) + Pass 2 (BigQuery Native)

Runs inside `try ... finally: bpd.close_session()` so all temporary session-scoped `_anonymous_*` tables/routines are cleaned up immediately upon completion or error.


In [ ]:
governance_report: dict[str, Any] = {}
pass1_biglake_records: list[dict[str, Any]] = []
pass2_native_records: list[dict[str, Any]] = []

try:
    governance_report = verify_slot_reservation_governance(
        client=bq_client,
        project_id=PROJECT_ID,
        ds_bronze_native=DS_BRONZE_NATIVE,
        reservation_name=BQ_RESERVATION_NAME,
        info_schema_region=BQ_INFO_SCHEMA_REGION,
        allow_unreserved_bq=ALLOW_UNRESERVED_BQ,
    )

    pass1_biglake_records = run_medallion_pass(
        bf_session=session,
        client=bq_client,
        engine_variant="bigframes_biglake_gcs",
        storage_backend="biglake_gcs_parquet",
        bronze_ds=DS_BRONZE_BIGLAKE,
        silver_ds=DS_SILVER_BQ,
        gold_ds=DS_GOLD_BQ,
        suffix="biglake",
    )

    pass2_native_records = run_medallion_pass(
        bf_session=session,
        client=bq_client,
        engine_variant="bigframes_native_bq",
        storage_backend="bq_native_capacitor",
        bronze_ds=DS_BRONZE_NATIVE,
        silver_ds=DS_SILVER_BQ,
        gold_ds=DS_GOLD_BQ,
        suffix="native",
    )
finally:
    bpd.close_session()
    print("Closed BigFrames session cleanly (_anonymous_* session artifacts purged).")


### End-of-Run BigLake GCS vs. BigQuery Native Parity & Storage-Engine Summary

Validates **100% row-count equality** and **`±0.01%` `checksum_net_paid` tolerance** between `bigframes_biglake_gcs` and `bigframes_native_bq` across all executed stages, isolating pure storage-layer impact (`GCS Parquet via BigLake` vs. `BigQuery Native Capacitor`).


In [ ]:
all_records = pass1_biglake_records + pass2_native_records
summary_df = pd.DataFrame(
    [
        {
            "engine_variant": r["engine_variant"],
            "storage_backend": r["storage_backend"],
            "stage_id": r["stage_id"],
            "wall_clock_sec": round(float(r["wall_clock_sec"]), 3),
            "output_row_count": int(r["output_row_count"]),
            "sample_fraction": float(r["sample_fraction"]),
            "checksum_net_paid": float(r["checksum_net_paid"]),
            "bq_total_slot_ms": int(r["bq_total_slot_ms"]),
            "bq_bytes_processed": int(r["bq_bytes_processed"]),
            "est_list_cost_usd": (
                round(float(r["est_list_cost_usd"]), 6)
                if r.get("est_list_cost_usd") is not None
                else None
            ),
            "reservation_id": r["reservation_id"],
            "bi_engine_mode": r["bi_engine_mode"],
        }
        for r in all_records
    ]
)

display(summary_df)

# Fail-closed parity check between BigLake GCS and Native Capacitor outputs.
p1_by_stage = {r["stage_id"]: r for r in pass1_biglake_records}
p2_by_stage = {r["stage_id"]: r for r in pass2_native_records}
assert set(p1_by_stage) == set(p2_by_stage), (
    f"Stage mismatch between BigLake {sorted(p1_by_stage)} and Native {sorted(p2_by_stage)}"
)

parity_rows = []
for stage_id in sorted(p1_by_stage):
    r1, r2 = p1_by_stage[stage_id], p2_by_stage[stage_id]
    rows1, rows2 = int(r1["output_row_count"]), int(r2["output_row_count"])
    chk1, chk2 = float(r1["checksum_net_paid"]), float(r2["checksum_net_paid"])
    rel_diff = abs(chk1 - chk2) / max(abs(chk2), 1.0)
    assert rows1 == rows2, f"{stage_id} row count mismatch: biglake={rows1} vs native={rows2}"
    assert rel_diff <= 1e-4, (
        f"{stage_id} checksum mismatch exceeds 0.01%: biglake={chk1}, native={chk2}, rel_diff={rel_diff:.6f}"
    )

    dur1, dur2 = float(r1["wall_clock_sec"]), float(r2["wall_clock_sec"])
    slot1, slot2 = int(r1["bq_total_slot_ms"]), int(r2["bq_total_slot_ms"])
    parity_rows.append(
        {
            "stage_id": stage_id,
            "row_count": rows1,
            "checksum_rel_diff_pct": round(rel_diff * 100.0, 6),
            "biglake_sec": round(dur1, 2),
            "native_sec": round(dur2, 2),
            "native_speedup_x": round(dur1 / max(dur2, 1e-6), 2),
            "biglake_slot_ms": slot1,
            "native_slot_ms": slot2,
            "slot_ratio_biglake_over_native": round(slot1 / max(slot2, 1), 2),
            "native_bi_engine": r2["bi_engine_mode"],
        }
    )

# Verify Stage 2A vs Stage 2B exact parity on small_1g (100% deterministic sample).
if SCALE_PRESET == "small_1g" and "stage2b_silver_udf" in p2_by_stage:
    for label, by_stage in [("biglake", p1_by_stage), ("native", p2_by_stage)]:
        s2a, s2b = by_stage["stage2a_silver_vectorized"], by_stage["stage2b_silver_udf"]
        assert s2a["output_row_count"] == s2b["output_row_count"], (
            f"{label} Stage 2A vs 2B row mismatch on small_1g"
        )
        assert abs(s2a["checksum_net_paid"] - s2b["checksum_net_paid"]) / max(abs(s2a["checksum_net_paid"]), 1.0) <= 1e-4

parity_df = pd.DataFrame(parity_rows)
print("[PASS] 100% row-count parity and <=0.01% checksum parity verified across BigLake GCS and Native Capacitor.")
display(parity_df)

tot_bl_sec = sum(float(r["wall_clock_sec"]) for r in pass1_biglake_records)
tot_nat_sec = sum(float(r["wall_clock_sec"]) for r in pass2_native_records)
print(
    f"End-to-End Medallion Totals ({SCALE_PRESET}): "
    f"bigframes_biglake_gcs={tot_bl_sec:.2f}s vs bigframes_native_bq={tot_nat_sec:.2f}s "
    f"(Storage speedup={tot_bl_sec / max(tot_nat_sec, 1e-6):.2f}x)"
)
